# 🏥 MediCare Patient Follow-Up Agent

**Domain:** Healthcare | **Role:** Data Scientist | **Format:** Jupyter Notebook + CSV

---

## Scenario
MediCare Clinic manages over 1,000 chronic disease patients across conditions like Type 2 Diabetes, Hypertension, Heart Failure, COPD, and Anemia. The clinical team needs an AI-powered agent that can:
- Autonomously review patient records
- Identify patients at clinical risk
- Generate prioritised follow-up care action plans

This notebook implements a **prototype Agentic AI system** using the **Groq LLM API** (`llama-3.3-70b-versatile`) with native tool-calling to orchestrate clinical decision-making.

### Notebook Structure
| Task | Description |
|------|-------------|
| Task 1 | Data Loading & Exploratory Analysis |
| Task 2 | Define Agent Tools |
| Task 3 | Implement the Agentic Loop |
| Task 4 | Single Patient Analysis |
| Task 5 | Missed Appointment Follow-Up |

---

## ⚙️ Setup & Configuration

In [34]:
pip install groq

Note: you may need to restart the kernel to use updated packages.


In [ ]:
# Install required packages if not already installed
# !pip install pandas groq -q

import pandas as pd
import json
from groq import Groq

# ── Groq API Configuration ────────────────────────────────────────────────
# llama-3.3-70b-versatile: best free model on Groq with native tool-calling
GROQ_API_KEY = "YOUR_GROQ_API_KEY_HERE"  # Replace with your key
MODEL        = "openai/gpt-oss-120b"

client = Groq(api_key=GROQ_API_KEY)
print(f"✅ Groq client initialised | Model: {MODEL}")

✅ Groq client initialised | Model: openai/gpt-oss-120b


---

## Task 1 — Data Loading & Exploratory Analysis

Load `patient_data.csv` and perform EDA to understand dataset structure, patient demographics, clinical distributions, and appointment adherence.

In [24]:
# ── Load patient dataset ─────────────────────────────────────────────────
df = pd.read_csv("patient_data.csv", parse_dates=["last_visit_date", "next_scheduled_visit"])
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head(3)

Shape: (100, 18)
Columns: ['patient_id', 'patient_name', 'age', 'gender', 'diagnosis', 'current_medication', 'lab_test', 'lab_value', 'lab_unit', 'last_visit_date', 'next_scheduled_visit', 'days_since_last_visit', 'missed_last_appointment', 'vitals_bp_systolic', 'vitals_bp_diastolic', 'vitals_heart_rate', 'vitals_spo2', 'notes']


,patient_id,patient_name,age,gender,diagnosis,current_medication,lab_test,lab_value,lab_unit,last_visit_date,next_scheduled_visit,days_since_last_visit,missed_last_appointment,vitals_bp_systolic,vitals_bp_diastolic,vitals_heart_rate,vitals_spo2,notes
0,P0001,Aarav Sharma,68,Male,Type 2 Diabetes,Metformin 500mg,HbA1c,9.9,%,2024-04-24,2024-06-08,251,No,174,70,95,96,Patient reports good medication adherence.
1,P0002,Priya Patel,29,Male,Heart Failure,Furosemide 40mg,BNP,317.6,pg/mL,2024-01-14,2024-02-28,352,No,174,91,72,97,Patient reports no new symptoms.
2,P0003,Rohan Mehta,45,Male,"Type 2 Diabetes, CKD","Insulin Glargine 10U, Losartan 50mg",HbA1c,11.4,%,2024-08-04,2024-10-03,149,No,132,86,64,91,Medication dose adjusted this visit.


In [25]:
# ── Exploratory Data Analysis ────────────────────────────────────────────
print("=" * 55)
print(" DATASET OVERVIEW")
print("=" * 55)

# Missing values
missing = df.isnull().sum()
print(f"\nMissing values: {missing[missing > 0].to_dict() or 'None — dataset complete'}")

# Demographics
print(f"\nAge  : mean={df['age'].mean():.1f}, min={df['age'].min()}, max={df['age'].max()}")
print(f"Gender: {df['gender'].value_counts().to_dict()}")

# Extract primary diagnosis (first item in compound diagnoses like 'Type 2 Diabetes, CKD')
df['primary_diagnosis'] = df['diagnosis'].apply(lambda x: x.split(',')[0].strip())
print(f"\nPrimary diagnoses:\n{df['primary_diagnosis'].value_counts().to_string()}")

# Appointment adherence
mc = df['missed_last_appointment'].value_counts()
print(f"\nMissed appointment → Yes: {mc.get('Yes', 0)}, No: {mc.get('No', 0)}")

# Visit timing
print(f"\nDays since last visit: mean={df['days_since_last_visit'].mean():.1f}, "
      f"max={df['days_since_last_visit'].max()}")

# Vitals
print("\nVitals Summary:")
print(df[["vitals_bp_systolic","vitals_bp_diastolic","vitals_heart_rate","vitals_spo2"]]
      .describe().round(1).to_string())

# Lab tests
print(f"\nUnique lab tests: {sorted(df['lab_test'].unique())}")

 DATASET OVERVIEW

Missing values: None — dataset complete

Age  : mean=55.4, min=28, max=82
Gender: {'Male': 53, 'Female': 47}

Primary diagnoses:
primary_diagnosis
Hypertension              18
Type 2 Diabetes           17
COPD                      13
Heart Failure             12
Hypothyroidism             9
Osteoarthritis             7
Depression                 7
Asthma                     6
Anemia                     6
Chronic Kidney Disease     5

Missed appointment → Yes: 15, No: 85

Days since last visit: mean=211.4, max=360

Vitals Summary:
       vitals_bp_systolic  vitals_bp_diastolic  vitals_heart_rate  vitals_spo2
count               100.0                100.0              100.0        100.0
mean                145.6                 87.7               82.4         94.5
std                  20.4                 13.6               14.5          2.8
min                 105.0                 65.0               58.0         90.0
25%                 130.0                 75.8    

In [26]:
# ── Clinical Risk Thresholds (evidence-based, per medical guidelines) ─────
# Used by the flag_clinical_risk tool to categorise lab values.
# The LLM agent makes final decisions — no hardcoded action logic.
RISK_THRESHOLDS = {
    "HbA1c"      : {"critical": 10.0, "high": 8.0,  "unit": "%",      "direction": "above"},
    "BNP"         : {"critical": 900,  "high": 400,  "unit": "pg/mL",  "direction": "above"},
    "eGFR"        : {"critical": 30,   "high": 45,   "unit": "mL/min", "direction": "below"},
    "BP Systolic" : {"critical": 180,  "high": 140,  "unit": "mmHg",   "direction": "above"},
    "FEV1%"       : {"critical": 30,   "high": 50,   "unit": "%",      "direction": "below"},
    "Hemoglobin"  : {"critical": 7.0,  "high": 8.0,  "unit": "g/dL",   "direction": "below"},
    "PHQ-9 Score" : {"critical": 20,   "high": 15,   "unit": "/27",    "direction": "above"},
    "Peak Flow"   : {"critical": 200,  "high": 320,  "unit": "L/min",  "direction": "below"},
    "Pain Score"  : {"critical": 8,    "high": 6,    "unit": "/10",    "direction": "above"},
}
print(f"✅ Risk thresholds configured for {len(RISK_THRESHOLDS)} lab test types")

✅ Risk thresholds configured for 9 lab test types


---

## Task 2 — Define Agent Tools

Define **4 tools** the LLM agent can autonomously call:
1. `get_patient_record` — Retrieve full patient record by ID
2. `flag_clinical_risk` — Assess risk level (LOW / HIGH / CRITICAL)
3. `generate_followup_action` — Create prioritised care action plan
4. `get_missed_appointment_patients` — List all missed-appointment patients

Each tool has:
- A **Python implementation** that queries the live DataFrame
- A **JSON schema** that the LLM uses to understand how to call the tool

In [27]:
# ── Tool 1: get_patient_record ───────────────────────────────────────────
def get_patient_record(patient_id: str) -> dict:
    """Retrieve all clinical data for a patient by their ID."""
    row = df[df["patient_id"] == patient_id]
    if row.empty:
        return {"error": f"Patient {patient_id} not found."}
    record = row.iloc[0].to_dict()
    # Convert Timestamp objects to ISO strings for JSON serialisation
    for k, v in record.items():
        if hasattr(v, "strftime"):
            record[k] = v.strftime("%Y-%m-%d")
    return record


# ── Tool 2: flag_clinical_risk ───────────────────────────────────────────
def flag_clinical_risk(patient_id: str) -> dict:
    """Assess clinical risk (LOW/HIGH/CRITICAL) from lab values, vitals, visit history."""
    record = get_patient_record(patient_id)
    if "error" in record:
        return record
    flags = []
    lab_test  = record.get("lab_test", "")
    lab_value = float(record.get("lab_value", 0))
    thresh    = RISK_THRESHOLDS.get(lab_test, {})
    if thresh:
        d = thresh["direction"]
        if d == "above":
            if lab_value >= thresh["critical"]:
                flags.append(f"CRITICAL: {lab_test}={lab_value}{record['lab_unit']} (>={thresh['critical']})")
            elif lab_value >= thresh["high"]:
                flags.append(f"HIGH: {lab_test}={lab_value}{record['lab_unit']} (>={thresh['high']})")
        else:
            if lab_value <= thresh["critical"]:
                flags.append(f"CRITICAL: {lab_test}={lab_value}{record['lab_unit']} (<={thresh['critical']})")
            elif lab_value <= thresh["high"]:
                flags.append(f"HIGH: {lab_test}={lab_value}{record['lab_unit']} (<={thresh['high']})")
    if record.get("missed_last_appointment") == "Yes":
        flags.append("MISSED: Patient missed their last scheduled appointment")
    if record.get("days_since_last_visit", 0) > 180:
        flags.append(f"OVERDUE: Last visit was {record['days_since_last_visit']} days ago")
    if record.get("vitals_bp_systolic", 0) >= 180:
        flags.append(f"CRITICAL: BP Systolic={record['vitals_bp_systolic']} mmHg — hypertensive crisis risk")
    if record.get("vitals_spo2", 100) < 92:
        flags.append(f"HIGH: SpO2={record['vitals_spo2']}% — below safe threshold")
    risk_level = ("CRITICAL" if any("CRITICAL" in f for f in flags)
                  else "HIGH" if flags else "LOW")
    return {"patient_id": patient_id, "patient_name": record["patient_name"],
            "diagnosis": record["diagnosis"], "risk_level": risk_level,
            "flags": flags or ["No critical flags detected"]}


# ── Tool 3: generate_followup_action ────────────────────────────────────
def generate_followup_action(patient_id: str, risk_level: str, flags: list) -> dict:
    """Generate a structured follow-up care action plan based on risk level and clinical flags."""
    record = get_patient_record(patient_id)
    if "error" in record:
        return record
    plan_map = {
        "CRITICAL": {"urgency": "Immediate (24 hours)",     "action": "Emergency outreach + urgent clinic appointment", "follow_up": "1-3 days"},
        "HIGH"    : {"urgency": "High priority (3-5 days)", "action": "Phone consultation + expedited appointment",     "follow_up": "1-2 weeks"},
        "LOW"     : {"urgency": "Routine",                  "action": "Standard monitoring at next scheduled visit",   "follow_up": "As scheduled"}
    }
    p = plan_map.get(risk_level, plan_map["LOW"])
    return {"patient_id": patient_id, "patient_name": record["patient_name"],
            "age": record["age"], "diagnosis": record["diagnosis"],
            "current_medication": record["current_medication"],
            "risk_level": risk_level, "clinical_flags": flags,
            "urgency": p["urgency"], "recommended_action": p["action"],
            "follow_up_in": p["follow_up"], "clinical_notes": record.get("notes", "")}


# ── Tool 4: get_missed_appointment_patients ──────────────────────────────
def get_missed_appointment_patients() -> dict:
    """Retrieve all patients who missed their last scheduled appointment."""
    cols = ["patient_id","patient_name","age","diagnosis","days_since_last_visit","next_scheduled_visit"]
    missed = df[df["missed_last_appointment"] == "Yes"][cols].copy()
    missed["next_scheduled_visit"] = missed["next_scheduled_visit"].astype(str)
    return {"count": len(missed), "patients": missed.to_dict(orient="records")}


print("✅ 4 tools implemented: get_patient_record, flag_clinical_risk, generate_followup_action, get_missed_appointment_patients")

✅ 4 tools implemented: get_patient_record, flag_clinical_risk, generate_followup_action, get_missed_appointment_patients


In [28]:
# ── JSON Schemas for Groq API tool-calling ───────────────────────────────
# These tell the LLM what tools exist, their purpose, and expected parameters.
TOOLS = [
    {"type": "function", "function": {
        "name": "get_patient_record",
        "description": "Retrieve complete clinical data for a patient using their patient_id.",
        "parameters": {"type": "object",
            "properties": {"patient_id": {"type": "string", "description": "Patient ID e.g. P0001"}},
            "required": ["patient_id"]}}},
    {"type": "function", "function": {
        "name": "flag_clinical_risk",
        "description": "Assess clinical risk level (LOW/HIGH/CRITICAL) from lab values, vitals, visit history.",
        "parameters": {"type": "object",
            "properties": {"patient_id": {"type": "string"}},
            "required": ["patient_id"]}}},
    {"type": "function", "function": {
        "name": "generate_followup_action",
        "description": "Generate a structured follow-up care action plan for a patient given risk level and flags.",
        "parameters": {"type": "object",
            "properties": {
                "patient_id" : {"type": "string"},
                "risk_level" : {"type": "string", "enum": ["LOW","HIGH","CRITICAL"]},
                "flags"      : {"type": "array", "items": {"type": "string"}}},
            "required": ["patient_id","risk_level","flags"]}}},
    {"type": "function", "function": {
        "name": "get_missed_appointment_patients",
        "description": "Retrieve all patients who missed their last appointment for outreach prioritisation.",
        "parameters": {"type": "object", "properties": {}, "required": []}}}
]

# Dispatch map: maps tool name strings to their Python implementations
TOOL_DISPATCH = {
    "get_patient_record"              : get_patient_record,
    "flag_clinical_risk"              : flag_clinical_risk,
    "generate_followup_action"        : generate_followup_action,
    "get_missed_appointment_patients" : get_missed_appointment_patients
}
print("✅ Tool schemas and dispatch map ready")

✅ Tool schemas and dispatch map ready


---

## Task 3 — Implement the Agentic Loop

Implements a **ReAct-style** agentic loop (Reason → Act → Observe → Repeat):

```
User Query
    ↓
[LLM] → decides tool to call
    ↓
[Tool] → executes, returns result
    ↓
[LLM] → reasons over result → decides next tool / final answer
    ↓
Final Response
```

In [29]:
# ── System Prompt ────────────────────────────────────────────────────────
SYSTEM_PROMPT = """You are MediCare AI, a clinical decision-support agent for MediCare Clinic.

Your responsibilities:
- Analyse patient records using the available tools
- Identify patients at HIGH or CRITICAL clinical risk
- Generate clear, actionable follow-up care plans for the care coordination team
- Always use tools to retrieve real data — do NOT invent clinical information
- Be structured and clinically precise

Risk guidance: CRITICAL = act within 24 hours | HIGH = act within 3-5 days | LOW = routine monitoring"""


def run_agent(user_query: str, verbose: bool = True) -> str:
    """
    Execute the ReAct-style agentic loop for a given query.

    Steps per iteration:
      1. Send conversation history + tools to LLM
      2. Execute any tool calls the LLM requests
      3. Append tool results to conversation
      4. Repeat until LLM produces a final text response (no tool calls)

    Args:
        user_query : Clinical task or question
        verbose    : If True, prints each tool call + result preview
    Returns:
        Final agent response string
    """
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": user_query}
    ]

    for iteration in range(10):  # Safety cap: max 10 LLM round-trips
        # Step 1: Call LLM with available tools
        response   = client.chat.completions.create(
            model=MODEL, messages=messages, tools=TOOLS,
            tool_choice="auto", temperature=0.1, max_tokens=2048
        )
        msg        = response.choices[0].message
        tool_calls = msg.tool_calls

        # Append assistant message to conversation history
        messages.append({
            "role": "assistant",
            "content": msg.content or "",
            "tool_calls": [{"id": tc.id, "type": "function",
                             "function": {"name": tc.function.name,
                                          "arguments": tc.function.arguments}}
                            for tc in tool_calls] if tool_calls else []
        })

        # Step 2: No tool calls = LLM has its final answer
        if not tool_calls:
            return msg.content

        # Step 3: Execute each tool and feed result back to LLM
        for tc in tool_calls:
            fn_name   = tc.function.name
            fn_args   = json.loads(tc.function.arguments)
            if verbose:
                print(f"  [Tool {iteration+1}] {fn_name}({fn_args})")
            fn_result = TOOL_DISPATCH[fn_name](**fn_args)
            if verbose:
                print(f"  [Result] {str(fn_result)[:250]}\n")
            messages.append({"role": "tool", "tool_call_id": tc.id,
                              "content": json.dumps(fn_result)})

    return "Agent reached maximum iterations without a final answer."


print("✅ Agentic loop defined — ready to run queries")

✅ Agentic loop defined — ready to run queries


---

## Task 4 — Single Patient Analysis

Run the agent on individual patients. The agent autonomously:
1. Retrieves the patient's full clinical record
2. Flags risk level with supporting evidence (lab values, vitals, visit history)
3. Generates a tailored, prioritised follow-up action plan

In [30]:
# ── Analysis: P0003 — Rohan Mehta, 45M, Type 2 Diabetes + CKD, HbA1c=11.4% ─
print("=" * 60)
print(" SINGLE PATIENT ANALYSIS — P0003")
print("=" * 60 + "\n")

result_p0003 = run_agent(
    "Perform a complete clinical review for patient P0003: "
    "retrieve their record, assess clinical risk, and generate a prioritised follow-up action plan.",
    verbose=True
)

print("\n" + "-" * 60)
print("AGENT RESPONSE — P0003")
print("-" * 60)
print(result_p0003)

 SINGLE PATIENT ANALYSIS — P0003

  [Tool 1] get_patient_record({'patient_id': 'P0003'})
  [Result] {'patient_id': 'P0003', 'patient_name': 'Rohan Mehta', 'age': 45, 'gender': 'Male', 'diagnosis': 'Type 2 Diabetes, CKD', 'current_medication': 'Insulin Glargine 10U, Losartan 50mg', 'lab_test': 'HbA1c', 'lab_value': 11.4, 'lab_unit': '%', 'last_visit

  [Tool 2] flag_clinical_risk({'patient_id': 'P0003'})
  [Result] {'patient_id': 'P0003', 'patient_name': 'Rohan Mehta', 'diagnosis': 'Type 2 Diabetes, CKD', 'risk_level': 'CRITICAL', 'flags': ['CRITICAL: HbA1c=11.4% (>=10.0)', 'HIGH: SpO2=91% — below safe threshold']}

  [Tool 3] generate_followup_action({'flags': ['CRITICAL: HbA1c=11.4% (>=10.0)', 'HIGH: SpO2=91% — below safe threshold'], 'patient_id': 'P0003', 'risk_level': 'CRITICAL'})
  [Result] {'patient_id': 'P0003', 'patient_name': 'Rohan Mehta', 'age': 45, 'diagnosis': 'Type 2 Diabetes, CKD', 'current_medication': 'Insulin Glargine 10U, Losartan 50mg', 'risk_level': 'CRITICAL', 'cl

In [31]:
# ── Analysis: P0008 — Deepa Krishnan, Heart Failure, BNP=1100 pg/mL (critical) ─
print("=" * 60)
print(" SINGLE PATIENT ANALYSIS — P0008")
print("=" * 60 + "\n")

result_p0008 = run_agent(
    "Perform a clinical risk assessment for patient P0008 who has Heart Failure. "
    "Retrieve their record, flag risk, and generate a detailed follow-up action plan.",
    verbose=True
)

print("\n" + "-" * 60)
print("AGENT RESPONSE — P0008")
print("-" * 60)
print(result_p0008)

 SINGLE PATIENT ANALYSIS — P0008

  [Tool 1] get_patient_record({'patient_id': 'P0008'})
  [Result] {'patient_id': 'P0008', 'patient_name': 'Deepa Krishnan', 'age': 45, 'gender': 'Male', 'diagnosis': 'Heart Failure', 'current_medication': 'Furosemide 40mg', 'lab_test': 'BNP', 'lab_value': 1100.1, 'lab_unit': 'pg/mL', 'last_visit_date': '2024-06-10'

  [Tool 2] flag_clinical_risk({'patient_id': 'P0008'})
  [Result] {'patient_id': 'P0008', 'patient_name': 'Deepa Krishnan', 'diagnosis': 'Heart Failure', 'risk_level': 'CRITICAL', 'flags': ['CRITICAL: BNP=1100.1pg/mL (>=900)', 'OVERDUE: Last visit was 204 days ago']}

  [Tool 3] generate_followup_action({'flags': ['CRITICAL: BNP=1100.1pg/mL (>=900)', 'OVERDUE: Last visit was 204 days ago'], 'patient_id': 'P0008', 'risk_level': 'CRITICAL'})
  [Result] {'patient_id': 'P0008', 'patient_name': 'Deepa Krishnan', 'age': 45, 'diagnosis': 'Heart Failure', 'current_medication': 'Furosemide 40mg', 'risk_level': 'CRITICAL', 'clinical_flags': ['CRITICA

---

## Task 5 — Missed Appointment Follow-Up

The agent identifies **all patients who missed their last appointment** and produces a prioritised outreach plan ranked by clinical urgency (CRITICAL > HIGH > LOW).

In [32]:
# ── Missed Appointment Bulk Follow-Up ───────────────────────────────────
print("=" * 60)
print(" MISSED APPOINTMENT FOLLOW-UP CAMPAIGN")
print("=" * 60 + "\n")

result_missed = run_agent(
    "Identify all patients who missed their last appointment. "
    "For each patient retrieve their record and flag clinical risk. "
    "Rank patients CRITICAL > HIGH > LOW and provide a structured outreach action plan "
    "the care coordination team can act on today.",
    verbose=True
)

print("\n" + "-" * 60)
print("MISSED APPOINTMENT OUTREACH PLAN")
print("-" * 60)
print(result_missed)

 MISSED APPOINTMENT FOLLOW-UP CAMPAIGN

  [Tool 1] get_missed_appointment_patients({})
  [Result] {'count': 15, 'patients': [{'patient_id': 'P0018', 'patient_name': 'Lakshmi Menon', 'age': 36, 'diagnosis': 'Heart Failure', 'days_since_last_visit': 157, 'next_scheduled_visit': '2024-09-10'}, {'patient_id': 'P0031', 'patient_name': 'Harish Bajaj', 

  [Tool 2] get_patient_record({'patient_id': 'P0018'})
  [Result] {'patient_id': 'P0018', 'patient_name': 'Lakshmi Menon', 'age': 36, 'gender': 'Female', 'diagnosis': 'Heart Failure', 'current_medication': 'Furosemide 40mg', 'lab_test': 'BNP', 'lab_value': 952.3, 'lab_unit': 'pg/mL', 'last_visit_date': '2024-07-27'

  [Tool 3] flag_clinical_risk({'patient_id': 'P0018'})
  [Result] {'patient_id': 'P0018', 'patient_name': 'Lakshmi Menon', 'diagnosis': 'Heart Failure', 'risk_level': 'CRITICAL', 'flags': ['CRITICAL: BNP=952.3pg/mL (>=900)', 'MISSED: Patient missed their last scheduled appointment']}

  [Tool 4] get_patient_record({'patient_id': '

In [33]:
# ── Summary statistics for missed-appointment patients ──────────────────
missed_df = df[df['missed_last_appointment'] == 'Yes'].copy()
print(f"Total patients who missed appointment: {len(missed_df)}")
print(f"\nBreakdown by primary diagnosis:")
print(missed_df['primary_diagnosis'].value_counts().to_string())
print(f"\nAvg days since last visit: {missed_df['days_since_last_visit'].mean():.1f}")
print(f"Max days since last visit: {missed_df['days_since_last_visit'].max()}")

# Flag risk level for each missed-appointment patient (using our tool directly)
print("\nRisk assessment for each missed-appointment patient:")
print(f"{'Patient ID':<12} {'Name':<20} {'Risk':<10} {'Top Flag'}")
print("-" * 80)
for _, row in missed_df.iterrows():
    risk = flag_clinical_risk(row['patient_id'])
    print(f"{row['patient_id']:<12} {row['patient_name']:<20} {risk['risk_level']:<10} {risk['flags'][0]}")

Total patients who missed appointment: 15

Breakdown by primary diagnosis:
primary_diagnosis
Type 2 Diabetes           4
Heart Failure             3
Hypertension              3
Chronic Kidney Disease    1
Asthma                    1
Osteoarthritis            1
Depression                1
Hypothyroidism            1

Avg days since last visit: 233.1
Max days since last visit: 340

Risk assessment for each missed-appointment patient:
Patient ID   Name                 Risk       Top Flag
--------------------------------------------------------------------------------
P0018        Lakshmi Menon        CRITICAL   CRITICAL: BNP=952.3pg/mL (>=900)
P0031        Harish Bajaj         HIGH       MISSED: Patient missed their last scheduled appointment
P0036        Vandana Srivastava   CRITICAL   MISSED: Patient missed their last scheduled appointment
P0041        Tarun Mukherjee      HIGH       HIGH: BP Systolic=171.4mmHg (>=140)
P0043        Gaurav Dey           HIGH       HIGH: BP Systolic=163.8

---

## 📝 Summary

### What Was Built

| Component | Detail |
|-----------|--------|
| **LLM** | Groq `openai/gpt-oss-120b` — best free model with native tool-calling |
| **Tools** | 4 tools: record retrieval, risk flagging, action plan generation, missed appt lookup |
| **Agent Loop** | ReAct-style: LLM reasons → calls tools → gets results → reasons again |
| **Risk Engine** | Evidence-based thresholds for 9 lab types + vitals + visit history |
| **Output** | Prioritised, structured follow-up action plans per patient |

### Clinical Risk Thresholds Applied

| Condition | Lab Test | High Threshold | Critical Threshold |
|-----------|----------|---------------|--------------------|
| Type 2 Diabetes | HbA1c | ≥8% | ≥10% |
| Heart Failure | BNP | ≥400 pg/mL | ≥900 pg/mL |
| CKD | eGFR | ≤45 mL/min | ≤30 mL/min |
| Hypertension | BP Systolic | ≥140 mmHg | ≥180 mmHg |
| COPD | FEV1% | ≤50% | ≤30% |
| Anemia | Hemoglobin | ≤8 g/dL | ≤7 g/dL |
| Depression | PHQ-9 | ≥15/27 | ≥20/27 |
| Asthma | Peak Flow | ≤320 L/min | ≤200 L/min |
| Osteoarthritis | Pain Score | ≥6/10 | ≥8/10 |

### Key Design Decisions
- **No hardcoded action logic**: The LLM decides what actions to take — only data retrieval and risk categorisation are deterministic
- **Tool-first architecture**: All clinical data flows through well-defined tool interfaces, making the system auditable and extensible
- **ReAct loop**: The agent can chain multiple tool calls before arriving at a final recommendation, mimicking clinical reasoning
- **Safety cap**: Maximum 10 LLM iterations per query prevents infinite loops